# IPPU recalibración de producción + atemperar crecimiento industrial (BAU 2050)

**Dos objetivos:**

1. **Recalibrar la producción industrial de año base a output real de Marruecos.** Validación vs. fuentes (USGS Minerals Yearbook Morocco 2020-21, OCP, APC) mostró que cemento y metales calzan, pero minería y químicos están muy por debajo del output físico real (dominado por fosfato OCP), y vidrio/plástico/papel están descalibrados.
2. **Atemperar el crecimiento industrial (y eliminar el spike de 2050).** Dos palancas: (a) se elimina el uplift OCP del `demscalar_ippu_chemicals` (= 1.0 plano), y (b) se fijan **todas las elasticidades de producción al PIB en 0.5 en todos los periodos, salvo cemento** (se deja 0.3→0.4). Esto quita los uplifts de proyección (chemicals 0.8, mining 0.8, metals 0.7) y aplana el crecimiento. La banda BAU de químicos pasa de ~72 Mt en 2050 (3.5×) a **~28 Mt (~1.5×)**.

## Validación (sim año base 2021 vs. real)

| Categoría | Sim 2021 | Real Marruecos (fuente) | Objetivo |
|---|---:|---|---:|
| cemento | 14.8 Mt | ~12.5 Mt (USGS/APC) | **sin cambio** (calza) |
| metales | 2.07 Mt | ~2.0-2.5 Mt (USGS/worldsteel) | **sin cambio** (calza) |
| minería | 6.32 Mt | ~38 Mt roca fosfática + ~2 Mt otros (USGS/OCP) | **40 Mt** |
| químicos | 1.20 Mt | ác. fosfórico ~7 + fertilizantes ~11 (OCP) | **18 Mt** |
| vidrio | 3,202 t | ~0.2-0.4 Mt | **0.3 Mt** |
| plástico | 3.00 Mt | ~1.8 Mt | **1.8 Mt** |
| papel | 1.68 Mt | unos cientos de kt | **0.4 Mt** |

## Método: tonelaje real con emisiones intactas

La producción IPPU se proyecta como `prod = prodinit[tp=0] × crecimiento(elasticidad·PIB) × demscalar` (`ippu.py:project_industrial_production`). Reescalar `prodinit ×f` reescala toda la trayectoria por `f`.

Pero la producción alimenta flujos por-tonelada que generan emisiones:
- **Energía INEN** = `producción × consumpinit_inen_energy_tj_per_tonne_production` (`energy_consumption.py`, usa la intensidad de tp=0).
- **Emisiones de proceso IPPU** = `producción × ef_ippu_*_per_tonne_production_{cat}` (vidrio CO₂ 0.2 t/t, plástico CH₄).
- **Aguas residuales** = `producción × vol_ippu_{cat}_m3_ww_per_tonne_production`.

Para que **solo cambie el tonelaje reportado y las emisiones queden intactas (calibración SNBC preservada)**, co-escalamos **todas** las intensidades `*_per_tonne_production_{cat}` por `1/f`. Entonces `producción × intensidad` es invariante en cada periodo → energía, emisiones de proceso y aguas residuales sin cambio.

> **Nota / tradeoff.** Al co-escalar, las intensidades por-tonelada se vuelven *factores efectivos de reconciliación*, no intensidades físicas: el modelo conserva las emisiones calibradas al inventario nacional (NIR) mientras presenta el tonelaje real. Esto es deliberado — confiamos más en la calibración de emisiones (al NIR/SNBC) que en las cifras de producción originales.

**Entrada:** `sisepuede_raw_input_morocco_fuels_inen_ippu_scoe.csv` (última base simulada, run `2026-05-27T17;23;39`).
**Salida:** `sisepuede_raw_input_morocco_fuels_inen_ippu_scoe_prodcal.csv`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path('/Users/fabianfuentes/git/ssp_morocco')
INPUT_DIR = REPO / 'ssp_modeling' / 'input_data'

INPUT_FILE = INPUT_DIR / 'sisepuede_raw_input_morocco_fuels_inen_ippu_scoe.csv'
OUTPUT_FILE = INPUT_DIR / 'sisepuede_raw_input_morocco_fuels_inen_ippu_scoe_prodcal.csv'

pd.options.display.float_format = '{:,.4f}'.format
df = pd.read_csv(INPUT_FILE)
print(f'Loaded {INPUT_FILE.name}: {df.shape}  tp=[{df.time_period.min()}, {df.time_period.max()}]')

Loaded sisepuede_raw_input_morocco_fuels_inen_ippu_scoe.csv: (56, 2442)  tp=[0, 55]


## 1 · Knobs — objetivos de producción año base (2021)

`SCALE[cat] = TARGET_2021 / CURRENT_PROD_2021`, donde `CURRENT_PROD_2021` viene del último run
(`prod_ippu_{cat}_tonne` en 2021, run `2026-05-27T17;23;39`). Como `prod ∝ prodinit[0]`, escalar
`prodinit` por `SCALE` hace que la producción 2021 aterrice exactamente en el objetivo.

Ajusta `TARGET_2021` para iterar. Cemento y metales NO se tocan (ya calzan con lo real).

In [ ]:
# Producción simulada en 2021 (prod_ippu_<cat>_tonne, run 2026-05-27T17;23;39) -- referencia para el factor
CURRENT_PROD_2021 = {
    'mining':    6_317_285,
    'chemicals': 1_197_618,
    'glass':         3_202,
    'plastic':   2_999_144,
    'paper':     1_676_916,
}

# Objetivos de tonelaje año base 2021 (output físico real de Marruecos)
TARGET_2021 = {
    'mining':    40_000_000,   # roca fosfatica ~38 Mt (USGS/OCP) + ~2 Mt otros (barita, sal, metales)
    'chemicals': 18_000_000,   # acido fosforico ~7 Mt + fertilizantes ~11 Mt (OCP); excl. acido sulfurico intermedio
    'glass':        300_000,   # ~0.3 Mt
    'plastic':    1_800_000,   # ~1.8 Mt (real, vs 3.0 Mt sobreestimado)
    'paper':        400_000,   # unos cientos de kt (vs 1.68 Mt sobreestimado)
}

SCALE = {c: TARGET_2021[c] / CURRENT_PROD_2021[c] for c in TARGET_2021}

_tbl = pd.DataFrame({
    'current_prod_2021': CURRENT_PROD_2021,
    'target_2021':       TARGET_2021,
    'scale_f':           SCALE,
})
print(_tbl.to_string(formatters={
    'current_prod_2021': '{:,.0f}'.format,
    'target_2021':       '{:,.0f}'.format,
    'scale_f':           '{:.4f}'.format,
}))

          current_prod_2021 target_2021 scale_f
mining            6,317,285  40,000,000  6.3318
chemicals         1,197,618  18,000,000 15.0298
glass                 3,202     300,000 93.6914
plastic           2,999,144   1,800,000  0.6002
paper             1,676,916     400,000  0.2385


## 2 · Identificar columnas a escalar

Por cada categoría: la columna ancla `prodinit_ippu_{cat}_tonne` (×f) y **todas** las intensidades
`*_per_tonne_production*{cat}*` — energía (`consumpinit_inen_energy_*`), emisiones de proceso
(`ef_ippu_*`) y aguas residuales (`vol_ippu_*`) — (×1/f). Se excluyen explícitamente las variantes
`recycled_{cat}` (su producción no se reescala).

In [ ]:
def intensity_cols_for(cat):
    # Columnas de intensidad por-tonelada-de-produccion de la categoria (excl. recycled).
    out = []
    for c in df.columns:
        cl = c.lower()
        if f'recycled_{cat}' in cl:
            continue
        if cat not in cl:
            continue
        if 'per_tonne_production' in cl:
            out.append(c)
    return sorted(out)

PRODINIT = {c: f'prodinit_ippu_{c}_tonne' for c in SCALE}
INTENSITY = {c: intensity_cols_for(c) for c in SCALE}

for c in SCALE:
    assert PRODINIT[c] in df.columns, f'falta {PRODINIT[c]}'
    print(f'[{c}]  f={SCALE[c]:.4f}')
    print(f'    prodinit: {PRODINIT[c]} = {df[df.time_period==0][PRODINIT[c]].iloc[0]:,.2f}')
    for icol in INTENSITY[c]:
        print(f'    intensity x1/f: {icol} = {df[df.time_period==0][icol].iloc[0]}')

[mining]  f=6.3318
    prodinit: prodinit_ippu_mining_tonne = 5,949,896.86
    intensity x1/f: consumpinit_inen_energy_tj_per_tonne_production_mining = 0.000133062
[chemicals]  f=15.0298
    prodinit: prodinit_ippu_chemicals_tonne = 1,127,968.91
    intensity x1/f: consumpinit_inen_energy_tj_per_tonne_production_chemicals = 0.018809423
    intensity x1/f: ef_ippu_tonne_c2f6_per_tonne_production_chemicals = 0
    intensity x1/f: ef_ippu_tonne_c2h3f3_per_tonne_production_chemicals = 0
    intensity x1/f: ef_ippu_tonne_c2hf5_per_tonne_production_chemicals = 0
    intensity x1/f: ef_ippu_tonne_c3f8_per_tonne_production_chemicals = 0
    intensity x1/f: ef_ippu_tonne_c3hf7_per_tonne_production_chemicals = 0
    intensity x1/f: ef_ippu_tonne_c4f10_per_tonne_production_chemicals = 0
    intensity x1/f: ef_ippu_tonne_c4h5f5_per_tonne_production_chemicals = 0
    intensity x1/f: ef_ippu_tonne_c5f12_per_tonne_production_chemicals = 0
    intensity x1/f: ef_ippu_tonne_c6f14_per_tonne_production_c

    intensity x1/f: vol_ippu_paper_m3_ww_per_tonne_production = 19


## 3 · Aplicar recalibración de producción

`prodinit ×f` y cada intensidad `×1/f`, en **todos** los periodos (se preserva la forma temporal y,
crucialmente, el valor de tp=0 que es el que el modelo usa como ancla).

In [ ]:
df_new = df.copy()

for c in SCALE:
    f = SCALE[c]
    df_new[PRODINIT[c]] = df_new[PRODINIT[c]].to_numpy() * f
    for icol in INTENSITY[c]:
        df_new[icol] = df_new[icol].to_numpy() / f

print('Recalibracion aplicada. prodinit (tp=0) antes -> despues:')
for c in SCALE:
    a = df[df.time_period==0][PRODINIT[c]].iloc[0]
    b = df_new[df_new.time_period==0][PRODINIT[c]].iloc[0]
    print(f'  {c:10s}: {a:>14,.0f} -> {b:>14,.0f}  ({SCALE[c]:.3f}x)')

Recalibracion aplicada. prodinit (tp=0) antes -> despues:
  mining    :      5,949,897 ->     37,673,759  (6.332x)
  chemicals :      1,127,969 ->     16,953,186  (15.030x)
  glass     :         15,078 ->      1,412,642  (93.691x)
  plastic   :      2,899,949 ->      1,740,466  (0.600x)
  paper     :      1,665,147 ->        397,193  (0.239x)


## 4 · Primera palanca — quitar el uplift OCP del demscalar (`demscalar_ippu_chemicals` plano = 1.0)

Se **elimina por completo el uplift OCP del demscalar**: `demscalar_ippu_chemicals = 1.0` plano en todos
los periodos. Esto:

- **Elimina el spike** — al ser plano no hay escalón terminal en 2050.
- **Quita el doblaje exógeno** (×2.0 en 2050) que inflaba la banda.

Los valores históricos (tp 0–7) ya eran 1.0, así que la calibración no se altera. La segunda palanca
(aplanar las elasticidades a 0.5) va en §5.

> **Nota.** A diferencia de la recalibración de §3 (neutra en emisiones), bajar el demscalar y las
> elasticidades **sí** reduce la producción futura y, por tanto, la energía INEN y emisiones de proceso
> en los años de proyección — justo lo deseado: menos producción ⇒ menos energía. La calibración
> histórica (tp 0–7) no se toca.

In [ ]:
DEMSCALAR_COL = 'demscalar_ippu_chemicals'
CHEM_DEMSCALAR_FLAT = 1.0   # Opcion A: se elimina el uplift OCP; quimicos crece solo con el PIB

# Plano en TODOS los periodos. Los valores historicos (tp 0-7) ya eran 1.0, asi que la
# calibracion no se altera; solo se aplana la trayectoria de proyeccion (antes rampa/escalon -> 2.0).
df_new[DEMSCALAR_COL] = float(CHEM_DEMSCALAR_FLAT)

snap = [7, 8, 15, 25, 33, 34, 35, 36, 45, 55]
cmp = pd.DataFrame({
    'year':       df.loc[df.time_period.isin(snap), 'year'].values,
    'demscalar_ANTES': df.loc[df.time_period.isin(snap), DEMSCALAR_COL].values,
    'demscalar_AHORA': df_new.loc[df_new.time_period.isin(snap), DEMSCALAR_COL].values,
}, index=[t for t in snap])
cmp.index.name = 'tp'
print(cmp.round(4))

    year  demscalar_ANTES  demscalar_AHORA
tp                                        
7   2022                1           1.0000
8   2023                1           1.0000
15  2030                1           1.0000
25  2040                1           1.0000
33  2048                1           1.0000
34  2049                1           1.0000
35  2050                2           1.0000
36  2051                2           1.0000
45  2060                2           1.0000
55  2070                2           1.0000


## 5 · Segunda palanca — elasticidades de producción uniformes a 0.5 (excepto cemento)

Todas las `elasticity_ippu_<cat>_production_to_gdp` se fijan en **0.5 en todos los periodos**, salvo
**cemento** (se deja 0.3→0.4, como pediste). Esto elimina los uplifts de proyección que metían las
libretas `*_bau_uplift` (chemicals 0.8, mining 0.8, metals 0.7) y deja un crecimiento uniforme ligado
al PIB. Para chemicals/mining/metals el periodo histórico ya estaba en 0.5, así que solo se aplana la
proyección; las demás ya estaban en 0.5 plano.

In [ ]:
ELAS_FLAT = 0.5
EXCLUDE_ELAS = {'cement'}   # cemento conserva 0.3 -> 0.4

elas_cols = sorted(c for c in df_new.columns
                   if c.startswith('elasticity_ippu_') and c.endswith('_production_to_gdp'))

print(f'{"categoria":22s}  antes tp0->tp35    despues tp0->tp35')
for c in elas_cols:
    cat = c.replace('elasticity_ippu_', '').replace('_production_to_gdp', '')
    b = df.sort_values('time_period')[c].to_numpy(dtype=float)
    if cat not in EXCLUDE_ELAS:
        df_new[c] = float(ELAS_FLAT)
    a = df_new.sort_values('time_period')[c].to_numpy(dtype=float)
    tag = '   <- sin cambio (cemento)' if cat in EXCLUDE_ELAS else ''
    print(f'{cat:22s}  {b[0]:.2f} -> {b[35]:.2f}      {a[0]:.2f} -> {a[35]:.2f}{tag}')

categoria               antes tp0->tp35    despues tp0->tp35
cement                  0.30 -> 0.40      0.30 -> 0.40   <- sin cambio (cemento)
chemicals               0.50 -> 0.80      0.50 -> 0.50
electronics             0.50 -> 0.50      0.50 -> 0.50
glass                   0.50 -> 0.50      0.50 -> 0.50
lime_and_carbonite      0.50 -> 0.50      0.50 -> 0.50
metals                  0.50 -> 0.70      0.50 -> 0.50
mining                  0.50 -> 0.80      0.50 -> 0.50
paper                   0.50 -> 0.50      0.50 -> 0.50
plastic                 0.50 -> 0.50      0.50 -> 0.50
rubber_and_leather      0.50 -> 0.50      0.50 -> 0.50
textiles                0.50 -> 0.50      0.50 -> 0.50
wood                    0.50 -> 0.50      0.50 -> 0.50


## 6 · Verificación

1. **Invariante de emisiones/energía** — para cada categoría e intensidad, `prodinit × intensidad` debe
   ser idéntico antes/después en **todos** los periodos (⇒ energía INEN, emisiones de proceso IPPU y
   aguas residuales sin cambio).
2. **demscalar plano** — `demscalar_ippu_chemicals` = 1.0 en todos los periodos (sin spike y sin uplift
   OCP).
3. **Elasticidades** — todas = 0.5 en todos los periodos, salvo cemento (0.3→0.4).
4. **Producción año base** — la nueva producción 2021 aterriza en los objetivos.

In [ ]:
# 1. Invariante prodinit x intensidad
print('=== 1. Invariante (prodinit x intensidad) - max diff relativa por columna ===')
max_rel = 0.0
for c in SCALE:
    for icol in INTENSITY[c]:
        old = df[PRODINIT[c]].to_numpy() * df[icol].to_numpy()
        new = df_new[PRODINIT[c]].to_numpy() * df_new[icol].to_numpy()
        denom = np.where(np.abs(old) > 0, np.abs(old), 1.0)
        rel = np.max(np.abs(new - old) / denom)
        max_rel = max(max_rel, rel)
        flag = 'OK' if rel < 1e-9 else 'XX'
        print(f'  [{flag}] {icol:55s} rel_diff={rel:.2e}')
assert max_rel < 1e-9, 'El invariante prodinit x intensidad se rompio.'
print(f'OK: emisiones/energia invariantes (max rel diff {max_rel:.2e}).')

# 2. demscalar plano = 1.0 (sin spike, sin uplift OCP)
print('\n=== 2. demscalar_ippu_chemicals plano = 1.0 ===')
v = df_new.sort_values('time_period')[DEMSCALAR_COL].to_numpy()
print(f'  min={v.min():.4f}  max={v.max():.4f}  (antes: 1.0 hasta 2049 -> 2.0 en 2050)')
assert np.allclose(v, 1.0), 'demscalar_ippu_chemicals no quedo plano en 1.0'
print('OK: sin spike y sin doblaje OCP.')

# 3. Elasticidades = 0.5 en todos los periodos (salvo cemento)
print('\n=== 3. elasticity_ippu_*_production_to_gdp ===')
bad = []
for c in elas_cols:
    cat = c.replace('elasticity_ippu_', '').replace('_production_to_gdp', '')
    vals = df_new[c].to_numpy(dtype=float)
    if cat in EXCLUDE_ELAS:
        ok = np.array_equal(vals, df[c].to_numpy(dtype=float))   # cemento sin cambio
        if not ok: bad.append((cat, 'cemento cambio'))
    else:
        ok = np.allclose(vals, 0.5)
        if not ok: bad.append((cat, f'no es 0.5 (min={vals.min()}, max={vals.max()})'))
assert not bad, f'Elasticidades mal seteadas: {bad}'
print('OK: todas = 0.5 en todos los periodos; cemento intacto (0.3 -> 0.4).')

# 4. Produccion ano base implicada
print('\n=== 4. Produccion 2021 implicada (current x f) ===')
for c in SCALE:
    implied = CURRENT_PROD_2021[c] * SCALE[c]
    print(f'  {c:10s}: {CURRENT_PROD_2021[c]:>14,.0f} -> {implied:>14,.0f}  (objetivo {TARGET_2021[c]:,.0f})')
print('\nCemento y metales: SIN CAMBIO en produccion base (no estan en SCALE).')

=== 1. Invariante (prodinit x intensidad) - max diff relativa por columna ===
  [OK] consumpinit_inen_energy_tj_per_tonne_production_mining  rel_diff=1.95e-16
  [OK] consumpinit_inen_energy_tj_per_tonne_production_chemicals rel_diff=1.71e-16
  [OK] ef_ippu_tonne_c2f6_per_tonne_production_chemicals       rel_diff=0.00e+00
  [OK] ef_ippu_tonne_c2h3f3_per_tonne_production_chemicals     rel_diff=0.00e+00
  [OK] ef_ippu_tonne_c2hf5_per_tonne_production_chemicals      rel_diff=0.00e+00
  [OK] ef_ippu_tonne_c3f8_per_tonne_production_chemicals       rel_diff=0.00e+00
  [OK] ef_ippu_tonne_c3hf7_per_tonne_production_chemicals      rel_diff=0.00e+00
  [OK] ef_ippu_tonne_c4f10_per_tonne_production_chemicals      rel_diff=0.00e+00
  [OK] ef_ippu_tonne_c4h5f5_per_tonne_production_chemicals     rel_diff=0.00e+00
  [OK] ef_ippu_tonne_c5f12_per_tonne_production_chemicals      rel_diff=0.00e+00
  [OK] ef_ippu_tonne_c6f14_per_tonne_production_chemicals      rel_diff=0.00e+00
  [OK] ef_ippu_tonne_cc4f8_pe

## 7 · Guardar

In [ ]:
df_new.to_csv(OUTPUT_FILE, index=False)
print(f'Wrote {OUTPUT_FILE}')
print(f'Shape: {df_new.shape}')

Wrote /Users/fabianfuentes/git/ssp_morocco/ssp_modeling/input_data/sisepuede_raw_input_morocco_fuels_inen_ippu_scoe_prodcal.csv
Shape: (56, 2442)


## 8 · Próximo paso

Correr el modelo apuntando a `sisepuede_raw_input_morocco_fuels_inen_ippu_scoe_prodcal.csv` y verificar
en el Tableau de "Industrial Production":

1. **Químicos sin spike y sin explosión** — la banda azul crece suave con el PIB (~28 Mt en 2050, ~1.5×
   vs 2025), en lugar de doblarse a ~72 Mt.
2. **Crecimiento aplanado en general** — minería y metales también crecen más despacio (elasticidad 0.5
   en vez de 0.8/0.7). Cemento mantiene su ritmo (0.3→0.4).
3. **Tonelaje realista de año base** — minería ~40 Mt, químicos ~18 Mt, vidrio ~0.3 Mt, plástico ~1.8 Mt,
   papel ~0.4 Mt en 2021.
4. **Emisiones** — la recalibración de §3 es neutra (mismas emisiones por el co-escalado). Las palancas
   de §4–§5 **sí** bajan la energía/emisiones futuras (hay menos producción hacia 2050), que es el efecto
   buscado.

**Para iterar:**
- *Tonelaje de año base:* ajusta `TARGET_2021` (§1).
- *Crecimiento:* `CHEM_DEMSCALAR_FLAT` (§4) y `ELAS_FLAT` / `EXCLUDE_ELAS` (§5). Para subir/bajar el
  ritmo de una categoría, edita su elasticidad o sácala de `EXCLUDE_ELAS`.